# Shield AI — Dataset Exploration

This notebook explores contract datasets for expanding Shield AI's corpus:

1. **Existing demo contracts** — baseline audit of what we have
2. **CUAD** (Contract Understanding Atticus Dataset) — 510 real contracts, 41 expert-annotated clause types, loaded directly from HuggingFace as CSV (no loading script issues)
3. **SEC EDGAR** — live API for public company material contracts
4. **Gap analysis** — what's missing vs Shield AI agent needs
5. **Synthetic data plan** — what to generate to fill the gaps
6. **Next steps** — which CUAD contracts to ingest first

**CUAD source**: `master_clauses.csv` from `huggingface.co/datasets/theatticusproject/cuad`  
510 rows × 83 columns — one row per contract, two columns per clause type (presence flag + extracted text)

## 0. Setup

In [1]:
import io
import json
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 60)

PROJECT_ROOT   = Path('..').resolve()
DEMO_DIR       = PROJECT_ROOT / 'demo_contracts'
NOTEBOOKS_DIR  = PROJECT_ROOT / 'notebooks'
DATA_DIR       = NOTEBOOKS_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data cache   : {DATA_DIR}')

Project root : /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI
Data cache   : /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data


---
## 1. Existing Demo Contracts — Baseline

In [2]:
demo_files = sorted(DEMO_DIR.glob('*.pdf')) + sorted(DEMO_DIR.glob('*.docx'))
print(f'Demo contracts on disk: {len(demo_files)}')
for f in demo_files:
    print(f'  {f.name:<42} {f.stat().st_size/1024:>6.1f} KB')

Demo contracts on disk: 7
  Clean_NDA.pdf                                 2.3 KB
  Healthcare_NoBAA.pdf                          2.5 KB
  Risky_Vendor.pdf                              4.4 KB
  SaaS_Standard.pdf                             3.0 KB
  Standard_Procurement.pdf                      2.5 KB
  Vendor_Agreement.pdf                          2.8 KB
  Vendor_Moderate.pdf                           2.7 KB


In [3]:
# Manual classification — what scenarios each file covers
existing = pd.DataFrame([
    {'file': 'Clean_NDA.pdf',            'type': 'NDA',         'risk': 'Low',    'compliance': 'None',       'security': 'Clean',     'agent_scenario': 'Happy path — auto-approve'},
    {'file': 'Vendor_Agreement.pdf',      'type': 'Vendor',      'risk': 'High',   'compliance': 'HIPAA',      'security': 'Injection', 'agent_scenario': 'Prompt injection → quarantine (Agent 4)'},
    {'file': 'Healthcare_NoBAA.pdf',      'type': 'Healthcare',  'risk': 'High',   'compliance': 'HIPAA',      'security': 'Clean',     'agent_scenario': 'Missing BAA → compliance fail (Agent 3)'},
    {'file': 'Risky_Vendor.pdf',          'type': 'Vendor',      'risk': 'High',   'compliance': 'None',       'security': 'Clean',     'agent_scenario': 'High risk → legal review (Agent 2)'},
    {'file': 'SaaS_Standard.pdf',         'type': 'SaaS',        'risk': 'Medium', 'compliance': 'GDPR',       'security': 'Clean',     'agent_scenario': 'Medium risk → manager review (Agent 5)'},
    {'file': 'Standard_Procurement.pdf',  'type': 'Procurement', 'risk': 'Low',    'compliance': 'None',       'security': 'Clean',     'agent_scenario': 'Standard → auto-approve (Agent 5)'},
    {'file': 'Vendor_Moderate.pdf',       'type': 'Vendor',      'risk': 'Medium', 'compliance': 'None',       'security': 'Clean',     'agent_scenario': 'Moderate risk → manager review (Agent 5)'},
])
display(existing)

# Coverage summary
print('\nTypes covered:', existing['type'].unique().tolist())
print('Risk levels  :', existing['risk'].value_counts().to_dict())
print('Compliance   :', existing['compliance'].unique().tolist())

,file,type,risk,compliance,security,agent_scenario
0,Clean_NDA.pdf,NDA,Low,None,Clean,Happy path — auto-approve
1,Vendor_Agreement.pdf,Vendor,High,HIPAA,Injection,Prompt injection → quarantine (Agent 4)
2,Healthcare_NoBAA.pdf,Healthcare,High,HIPAA,Clean,Missing BAA → compliance fail (Agent 3)
3,Risky_Vendor.pdf,Vendor,High,None,Clean,High risk → legal review (Agent 2)
4,SaaS_Standard.pdf,SaaS,Medium,GDPR,Clean,Medium risk → manager review (Agent 5)
5,Standard_Procurement.pdf,Procurement,Low,None,Clean,Standard → auto-approve (Agent 5)
6,Vendor_Moderate.pdf,Vendor,Medium,None,Clean,Moderate risk → manager review (Agent 5)



Types covered: ['NDA', 'Vendor', 'Healthcare', 'SaaS', 'Procurement']
Risk levels  : {'High': 3, 'Low': 2, 'Medium': 2}
Compliance   : ['None', 'HIPAA', 'GDPR']


In [4]:
print('=== GAPS in existing demo set ===')
gaps = [
    ('Government / FAR / DFARS',            'Agent 3 compliance — no federal procurement contracts'),
    ('Finance / PCI-DSS / SOX',             'Agent 3 compliance — no financial sector contracts'),
    ('Employment / Non-compete',             'Agent 7 Q&A — common clause category, zero coverage'),
    ('Multi-party agreements (3+ parties)', 'Agent 1 extraction — only 2-party contracts in demo'),
    ('Expired / past-termination contracts','Agent 1 extraction — date edge cases not tested'),
    ('$0 / uncapped liability',             'Agent 2 risk — extreme risk edge case not tested'),
    ('Amendment / addendum docs',           'Agent 1 extraction — cross-document references missing'),
    ('International / GDPR + CCPA combo',  'Agent 3 compliance — only single-framework contracts'),
    ('IP assignment / license grants',      'Agent 2/3 — no IP-focused contracts'),
    ('Supply chain / distribution',         'Agent 6 analytics — more vendor types needed for SQL variety'),
]
for gap, why in gaps:
    print(f'  ❌ {gap:<45} → {why}')

=== GAPS in existing demo set ===
  ❌ Government / FAR / DFARS                      → Agent 3 compliance — no federal procurement contracts
  ❌ Finance / PCI-DSS / SOX                       → Agent 3 compliance — no financial sector contracts
  ❌ Employment / Non-compete                      → Agent 7 Q&A — common clause category, zero coverage
  ❌ Multi-party agreements (3+ parties)           → Agent 1 extraction — only 2-party contracts in demo
  ❌ Expired / past-termination contracts          → Agent 1 extraction — date edge cases not tested
  ❌ $0 / uncapped liability                       → Agent 2 risk — extreme risk edge case not tested
  ❌ Amendment / addendum docs                     → Agent 1 extraction — cross-document references missing
  ❌ International / GDPR + CCPA combo             → Agent 3 compliance — only single-framework contracts
  ❌ IP assignment / license grants                → Agent 2/3 — no IP-focused contracts
  ❌ Supply chain / distribution                 

---
## 2. CUAD Dataset — 510 Real Contracts, 41 Clause Types

Loading `master_clauses.csv` directly from HuggingFace — avoids the broken loading script issue.  
Each row = 1 contract. Each clause type = 2 columns: `ClauseType` (contains? Yes/No) and `ClauseType-Answer` (extracted text).

In [5]:
CUAD_CSV_URL = (
    'https://huggingface.co/datasets/theatticusproject/cuad'
    '/resolve/main/CUAD_v1/master_clauses.csv'
)
CUAD_CACHE = DATA_DIR / 'master_clauses.csv'

if CUAD_CACHE.exists():
    print(f'Loading from cache: {CUAD_CACHE}')
    df = pd.read_csv(CUAD_CACHE, low_memory=False)
else:
    print('Downloading CUAD master_clauses.csv (~3.8 MB)...')
    r = requests.get(CUAD_CSV_URL, timeout=60)
    r.raise_for_status()
    CUAD_CACHE.write_bytes(r.content)
    df = pd.read_csv(io.BytesIO(r.content), low_memory=False)
    print(f'  Saved to {CUAD_CACHE}')

print(f'\nShape: {df.shape}  ({df.shape[0]} contracts × {df.shape[1]} columns)')
print(f'First 5 columns: {list(df.columns[:5])}')

Loading from cache: /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data/master_clauses.csv

Shape: (510, 83)  (510 contracts × 83 columns)
First 5 columns: ['Filename', 'Document Name', 'Document Name-Answer', 'Parties', 'Parties-Answer']


In [6]:
# Parse the column structure:
# Odd columns = clause type presence (Yes/No or NaN)
# Even columns = clause type answer text
# Column 0 = Filename

all_cols = list(df.columns)

# Clause type columns are those WITHOUT '-Answer' suffix (excluding 'Filename')
clause_cols   = [c for c in all_cols if not c.endswith('-Answer') and c != 'Filename']
answer_cols   = [c for c in all_cols if c.endswith('-Answer')]

print(f'Filename column : 1')
print(f'Clause presence : {len(clause_cols)} columns')
print(f'Clause answers  : {len(answer_cols)} columns')
print(f'\nAll {len(clause_cols)} clause types:')
for i, c in enumerate(clause_cols, 1):
    print(f'  {i:2d}. {c}')

Filename column : 1
Clause presence : 42 columns
Clause answers  : 40 columns

All 42 clause types:
   1. Document Name
   2. Parties
   3. Agreement Date
   4. Effective Date
   5. Expiration Date
   6. Renewal Term
   7. Notice Period To Terminate Renewal
   8. Notice Period To Terminate Renewal- Answer
   9. Governing Law
  10. Most Favored Nation
  11. Competitive Restriction Exception
  12. Non-Compete
  13. Exclusivity
  14. No-Solicit Of Customers
  15. No-Solicit Of Employees
  16. Non-Disparagement
  17. Termination For Convenience
  18. Rofr/Rofo/Rofn
  19. Change Of Control
  20. Anti-Assignment
  21. Revenue/Profit Sharing
  22. Price Restrictions
  23. Minimum Commitment
  24. Volume Restriction
  25. Ip Ownership Assignment
  26. Joint Ip Ownership
  27. License Grant
  28. Non-Transferable License
  29. Affiliate License-Licensor
  30. Affiliate License-Licensee
  31. Unlimited/All-You-Can-Eat-License
  32. Irrevocable Or Perpetual License
  33. Source Code Escrow
  34. 

In [7]:
# Compute presence rate for each clause type
# A clause is 'present' if the answer column has non-null, non-empty text
presence = {}
for cc, ac in zip(clause_cols, answer_cols):
    present_mask = df[ac].notna() & (df[ac].astype(str).str.strip() != '')
    presence[cc] = {
        'present': int(present_mask.sum()),
        'total':   len(df),
        'pct':     round(present_mask.mean() * 100, 1),
    }

df_presence = pd.DataFrame(presence).T.sort_values('pct', ascending=False).reset_index()
df_presence.columns = ['clause_type', 'present', 'total', 'pct']

print('=== Clause presence rates (top 15 most common) ===')
display(df_presence.head(15))
print(f'\n=== Clause presence rates (bottom 10 rarest) ===')
display(df_presence.tail(10))

=== Clause presence rates (top 15 most common) ===


,clause_type,present,total,pct
0,Document Name,510.0,510.0,100.0
1,Unlimited/All-You-Can-Eat-License,510.0,510.0,100.0
2,Volume Restriction,510.0,510.0,100.0
3,Ip Ownership Assignment,510.0,510.0,100.0
4,Joint Ip Ownership,510.0,510.0,100.0
5,License Grant,510.0,510.0,100.0
6,Non-Transferable License,510.0,510.0,100.0
7,Affiliate License-Licensor,510.0,510.0,100.0
8,Affiliate License-Licensee,510.0,510.0,100.0
9,Irrevocable Or Perpetual License,510.0,510.0,100.0



=== Clause presence rates (bottom 10 rarest) ===


,clause_type,present,total,pct
30,Non-Disparagement,510.0,510.0,100.0
31,Termination For Convenience,510.0,510.0,100.0
32,Rofr/Rofo/Rofn,510.0,510.0,100.0
33,Change Of Control,510.0,510.0,100.0
34,Parties,509.0,510.0,99.8
35,Agreement Date,465.0,510.0,91.2
36,Notice Period To Terminate Renewal,434.0,510.0,85.1
37,Effective Date,359.0,510.0,70.4
38,Expiration Date,329.0,510.0,64.5
39,Renewal Term,163.0,510.0,32.0


In [8]:
# Visualise clause presence rates
fig = px.bar(
    df_presence.sort_values('pct'),
    x='pct',
    y='clause_type',
    orientation='h',
    color='pct',
    color_continuous_scale='RdYlGn',
    title='CUAD: How often each clause type appears across 510 contracts (%)',
    labels={'pct': 'Contracts containing clause (%)', 'clause_type': ''},
    height=950,
    template='plotly_dark',
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

In [9]:
# How many of the 41 clauses does each contract have? (completeness)
def has_answer(row, ac):
    v = row[ac]
    return pd.notna(v) and str(v).strip() != ''

df['clauses_present'] = df.apply(
    lambda row: sum(has_answer(row, ac) for ac in answer_cols), axis=1
)
df['completeness_pct'] = (df['clauses_present'] / len(answer_cols) * 100).round(1)

print('=== Per-contract clause completeness ===')
print(df[['Filename', 'clauses_present', 'completeness_pct']].describe().round(1))

fig = px.histogram(
    df,
    x='completeness_pct',
    nbins=25,
    title='CUAD: Per-contract completeness — % of 41 clause types present',
    labels={'completeness_pct': 'Completeness (%)', 'count': 'Contracts'},
    template='plotly_dark',
)
median_val = df['completeness_pct'].median()
fig.add_vline(x=median_val, line_dash='dash',
              annotation_text=f'Median: {median_val:.0f}%', annotation_position='top right')
fig.show()

=== Per-contract clause completeness ===
       clauses_present  completeness_pct
count            510.0             510.0
mean              38.4              96.1
std                1.2               3.1
min               35.0              87.5
25%               38.0              95.0
50%               39.0              97.5
75%               39.0              97.5
max               40.0             100.0


In [10]:
# Infer contract types from filenames (CUAD uses descriptive filenames)
type_patterns = {
    'NDA / Confidentiality':   r'nda|confidential|non.disclos',
    'Software / SaaS License': r'software|licens|saas|subscript',
    'Service Agreement':       r'service|msa|master.service|professional',
    'Employment':              r'employ|contractor|consultant|staffing',
    'Supply / Vendor':         r'supply|vendor|supplier|purchas|procure',
    'Partnership / JV':        r'partner|joint.venture|collabor',
    'Distribution':            r'distribut|resell|channel',
    'IP / Assignment':         r'ip.assign|intellectual|patent|trademark|copyright',
    'Lease':                   r'lease|rental',
    'Government':              r'gov|federal|government|public',
}

def infer_type(filename: str) -> str:
    name = str(filename).lower()
    for label, pattern in type_patterns.items():
        if re.search(pattern, name):
            return label
    return 'Other / Mixed'

df['inferred_type'] = df['Filename'].apply(infer_type)
type_counts = df['inferred_type'].value_counts().reset_index()
type_counts.columns = ['type', 'count']

display(type_counts)

fig = px.pie(
    type_counts,
    values='count',
    names='type',
    title='CUAD: Inferred contract type distribution (510 contracts)',
    template='plotly_dark',
    hole=0.4,
)
fig.show()

,type,count
0,Other / Mixed,314
1,Software / SaaS License,45
2,Distribution,45
3,Service Agreement,38
4,Supply / Vendor,24
5,Partnership / JV,24
6,IP / Assignment,15
7,NDA / Confidentiality,5


In [11]:
# Sample clause texts — see what actual extracted answers look like
# These are the clauses most relevant to Shield AI agents
sample_clauses = [
    ('Cap on Liability',      'Cap On Liability-Answer'),
    ('Uncapped Liability',    'Uncapped Liability-Answer'),
    ('Governing Law',         'Governing Law-Answer'),
    ('Audit Rights',          'Audit Rights-Answer'),
    ('Anti-Assignment',       'Anti-Assignment-Answer'),
]

for label, col in sample_clauses:
    if col not in df.columns:
        # fuzzy match column
        matches = [c for c in df.columns if label.lower().replace(' ', '') in c.lower().replace(' ', '')]
        col = matches[0] if matches else None
    if not col:
        print(f'Column not found for {label}\n')
        continue
        
    samples = df[df[col].notna() & (df[col].str.strip() != '')][['Filename', col]].head(2)
    print(f'\n{'═'*70}')
    print(f'  {label.upper()}')
    print(f'{'═'*70}')
    for _, row in samples.iterrows():
        text = str(row[col])[:250]
        print(f'  Contract: {row["Filename"][:55]}')
        print(f'  Text    : "{text}..."' if len(str(row[col])) > 250 else f'  Text    : "{text}"')
        print()


══════════════════════════════════════════════════════════════════════
  CAP ON LIABILITY
══════════════════════════════════════════════════════════════════════
  Contract: CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10
  Text    : "Yes"

  Contract: EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B(01)_5251
  Text    : "Yes"


══════════════════════════════════════════════════════════════════════
  UNCAPPED LIABILITY
══════════════════════════════════════════════════════════════════════
  Contract: CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10
  Text    : "No"

  Contract: EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B(01)_5251
  Text    : "Yes"


══════════════════════════════════════════════════════════════════════
  GOVERNING LAW
══════════════════════════════════════════════════════════════════════
  Contract: CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10
  Text    : "Nevada"

  Contract: EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B(01)_5251
  Text    

In [12]:
# Map CUAD clause types to Shield AI agents
# This tells us which CUAD annotations we can use as eval ground truth

# Find matching CUAD columns for each agent's domain
def find_col(keyword: str, columns: list) -> str | None:
    kw = keyword.lower().replace(' ', '').replace('-', '')
    for c in columns:
        if kw in c.lower().replace(' ', '').replace('-', ''):
            return c
    return None

agent_clause_map = {
    'Agent 1 — Extraction': [
        'Document Name', 'Parties', 'Agreement Date', 'Effective Date',
        'Expiration Date', 'Governing Law', 'Renewal Term', 'Notice Period To Terminate Renewal',
    ],
    'Agent 2 — Risk Assessment': [
        'Cap On Liability', 'Uncapped Liability', 'Liquidated Damages',
        'Warranty Duration', 'Insurance', 'Minimum Commitment',
        'Anti-Assignment', 'Change Of Control', 'Covenant Not To Sue',
    ],
    'Agent 3 — Compliance': [
        'Audit Rights', 'IP Ownership Assignment', 'Joint Ip Ownership',
        'License Grant', 'Non-Transferable License', 'Source Code Escrow',
        'Post-Termination Services',
    ],
    'Agent 7 — Contract Q&A': [
        'Non-Compete', 'Exclusivity', 'No-Solicit Of Customers',
        'No-Solicit Of Employees', 'Non-Disparagement', 'Revenue/Profit Sharing',
        'Volume Restriction', 'Most Favored Nation', 'Price Restrictions',
        'Irrevocable Or Perpetual License', 'Unlimited/All-You-Can-Eat-License',
        'Rofo/Rofr/Rofn', 'Third Party Beneficiary',
    ],
}

print('=== CUAD → Shield AI agent mapping + coverage ===')
for agent, clauses in agent_clause_map.items():
    print(f'\n{agent}:')
    for c in clauses:
        row = df_presence[df_presence['clause_type'].str.lower() == c.lower()]
        pct = f"{row['pct'].values[0]:.0f}%" if len(row) else 'not in CUAD'
        status = '✅' if len(row) and row['pct'].values[0] > 20 else '⚠️ ' if len(row) else '❌'
        print(f'  {status} {c:<45} {pct} of contracts')

=== CUAD → Shield AI agent mapping + coverage ===

Agent 1 — Extraction:
  ✅ Document Name                                 100% of contracts
  ✅ Parties                                       100% of contracts
  ✅ Agreement Date                                91% of contracts
  ✅ Effective Date                                70% of contracts
  ✅ Expiration Date                               64% of contracts
  ✅ Governing Law                                 100% of contracts
  ✅ Renewal Term                                  32% of contracts
  ✅ Notice Period To Terminate Renewal            85% of contracts

Agent 2 — Risk Assessment:
  ✅ Cap On Liability                              100% of contracts
  ✅ Uncapped Liability                            100% of contracts
  ✅ Liquidated Damages                            100% of contracts
  ✅ Warranty Duration                             100% of contracts
  ✅ Insurance                                     100% of contracts
  ✅ Minimum Commitme

In [13]:
# Score each contract for Shield AI ingestion priority
# High score = many key clauses present + reasonable length

PRIORITY_CLAUSES = [
    'Cap On Liability-Answer', 'Uncapped Liability-Answer',
    'Governing Law-Answer',    'Audit Rights-Answer',
    'Anti-Assignment-Answer',  'Non-Compete-Answer',
    'IP Ownership Assignment-Answer', 'Change Of Control-Answer',
]

def count_priority_clauses(row):
    count = 0
    for col in PRIORITY_CLAUSES:
        if col in df.columns and pd.notna(row.get(col)) and str(row.get(col)).strip():
            count += 1
    return count

df['priority_score'] = df.apply(count_priority_clauses, axis=1)

# Estimate character count from the answers we have
df['approx_chars'] = df[answer_cols].apply(
    lambda row: sum(len(str(v)) for v in row if pd.notna(v)), axis=1
)

# Combined score: priority clauses × 3 + completeness bonus
df['shield_ai_score'] = (
    df['priority_score'] * 3 +
    df['completeness_pct'] * 0.5
).round(1)

top20 = (
    df[['Filename', 'inferred_type', 'clauses_present', 'priority_score',
        'completeness_pct', 'shield_ai_score']]
    .sort_values('shield_ai_score', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print('=== Top 20 CUAD contracts for Shield AI ingestion ===')
display(top20)

=== Top 20 CUAD contracts for Shield AI ingestion ===


,Filename,inferred_type,clauses_present,priority_score,completeness_pct,shield_ai_score
0,TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-10.1_Affiliate Agreement.pdf,Other / Mixed,40,7,100.0,71.0
1,ScansourceInc_20190822_10-K_EX-10.38_11793958_EX-10.38_Distributor Agreement1.pdf,Distribution,40,7,100.0,71.0
2,ENTERTAINMENTGAMINGASIAINC_02_15_2005-EX-10.5-DISTRIBUTOR AGREEMENT.PDF,Distribution,40,7,100.0,71.0
3,StampscomInc_20001114_10-Q_EX-10.47_2631630_EX-10.47_Co-Branding Agreement.pdf,Other / Mixed,40,7,100.0,71.0
4,"ETELOS,INC_03_09_2004-EX-10.8-DISTRIBUTOR AGREEMENT.PDF",Distribution,40,7,100.0,71.0
5,RandWorldwideInc_20010402_8-KA_EX-10.2_2102464_EX-10.2_Co-Branding Agreement.pdf,Other / Mixed,40,7,100.0,71.0
6,RaeSystemsInc_20001114_10-Q_EX-10.57_2631790_EX-10.57_Co-Branding Agreement.pdf,Other / Mixed,40,7,100.0,71.0
7,EUROPEANMICROHOLDINGSINC_03_06_1998-EX-10.6-DISTRIBUTOR AGREEMENT.PDF,Distribution,40,7,100.0,71.0
8,NeoformaInc_19991202_S-1A_EX-10.26_5224521_EX-10.26_Co-Branding Agreement.pdf,Other / Mixed,40,7,100.0,71.0
9,LeadersonlineInc_20000427_S-1A_EX-10.8_4991089_EX-10.8_Co-Branding Agreement.pdf,Other / Mixed,40,7,100.0,71.0


In [14]:
# Export top contracts metadata for the next notebook
top20_export = []
for _, row in top20.iterrows():
    # Collect clause answers for context
    clauses = {}
    for cc, ac in zip(clause_cols, answer_cols):
        val = row.get(ac)
        if pd.notna(val) and str(val).strip():
            clauses[cc] = str(val).strip()[:500]  # truncate for storage
    top20_export.append({
        'filename': row['Filename'],
        'inferred_type': row['inferred_type'],
        'clauses_present': int(row['clauses_present']),
        'priority_score': int(row['priority_score']),
        'shield_ai_score': float(row['shield_ai_score']),
        'key_clauses': clauses,
    })

out = DATA_DIR / 'cuad_top20_for_ingestion.json'
out.write_text(json.dumps(top20_export, indent=2))
print(f'Saved top 20 contracts to {out}')
print(f'\nType breakdown:')
for t, n in Counter(c['inferred_type'] for c in top20_export).most_common():
    print(f'  {t:<35} {n} contracts')

Saved top 20 contracts to /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data/cuad_top20_for_ingestion.json

Type breakdown:
  Other / Mixed                       9 contracts
  Distribution                        7 contracts
  Service Agreement                   2 contracts
  Software / SaaS License             1 contracts
  Partnership / JV                    1 contracts


---
## 3. SEC EDGAR — Live API for Public Company Contracts

In [15]:
# SEC requires a User-Agent header identifying who you are
EDGAR_HEADERS = {'User-Agent': 'ShieldAI-Research research@shieldai.example.com'}

def edgar_full_text_search(query: str, form: str = 'EX-10', n: int = 5) -> pd.DataFrame:
    """Full-text search across SEC filings."""
    url = 'https://efts.sec.gov/LATEST/search-index'
    params = {
        'q': f'"{query}"',
        'forms': form,
        'dateRange': 'custom',
        'startdt': '2023-01-01',
        'enddt': '2025-01-01',
    }
    try:
        r = requests.get(url, params=params, headers=EDGAR_HEADERS, timeout=12)
        hits = r.json().get('hits', {}).get('hits', [])
        rows = []
        for h in hits[:n]:
            s = h.get('_source', {})
            rows.append({
                'company':   s.get('entity_name', '?'),
                'form':      s.get('form_type', form),
                'filed':     s.get('file_date', ''),
                'file_num':  s.get('file_num', ''),
                'period':    s.get('period_of_report', ''),
            })
        return pd.DataFrame(rows) if rows else pd.DataFrame()
    except Exception as e:
        print(f'  ⚠️  EDGAR search error: {e}')
        return pd.DataFrame()


# EX-10 = material contract exhibits filed with 10-K / 10-Q reports
print('Searching EDGAR EX-10 exhibits (material contracts)...\n')

searches = {
    'SaaS / Software':       'software as a service subscription agreement',
    'Data Processing (DPA)': 'data processing agreement GDPR',
    'IP License':            'intellectual property license agreement',
    'Finance / Banking':     'credit facility agreement',
    'Employment / Exec':     'executive employment agreement compensation',
    'Supply Chain':          'master supply agreement vendor',
    'Government':            'federal contract agreement FAR clause',
    'Partnership':           'strategic partnership agreement collaboration',
}

edgar_results = {}
for label, query in searches.items():
    results = edgar_full_text_search(query, n=3)
    edgar_results[label] = results
    status = f'{len(results)} hits' if not results.empty else '0 hits'
    print(f'  {label:<30} → {status}')
    time.sleep(0.4)  # respect SEC rate limits

Searching EDGAR EX-10 exhibits (material contracts)...



  SaaS / Software                → 0 hits

  Data Processing (DPA)          → 0 hits


  IP License                     → 0 hits


  Finance / Banking              → 0 hits


  Employment / Exec              → 0 hits


  Supply Chain                   → 0 hits


  Government                     → 0 hits


  Partnership                    → 0 hits


In [16]:
# Show results for the most useful category
for label, df_edgar in edgar_results.items():
    if not df_edgar.empty:
        print(f'\n=== {label} ===')
        display(df_edgar)

In [17]:
# How to download an actual contract from EDGAR
# (demonstration — requires knowing the accession number)

def edgar_get_filing_documents(accession_no: str, cik: str) -> list[dict]:
    """Get list of documents in a specific filing."""
    clean_acc = accession_no.replace('-', '')
    url = f'https://www.sec.gov/Archives/edgar/full-index/'
    api_url = f'https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json'
    try:
        r = requests.get(api_url, headers=EDGAR_HEADERS, timeout=10)
        data = r.json()
        recent = data.get('filings', {}).get('recent', {})
        return [{
            'form': f,
            'date': d,
            'accession': a,
        } for f, d, a in zip(
            recent.get('form', []),
            recent.get('filingDate', []),
            recent.get('accessionNumber', []),
        ) if '10' in f or 'EX' in f][:10]
    except Exception as e:
        return [{'error': str(e)}]

# Example: Salesforce (CIK = 0001108524)
print('Example: Salesforce recent 10-K/EX filings')
sf_filings = edgar_get_filing_documents('', '1108524')
if sf_filings and 'error' not in sf_filings[0]:
    display(pd.DataFrame(sf_filings))
else:
    print(f'  Result: {sf_filings}')

Example: Salesforce recent 10-K/EX filings


,form,date,accession
0,10-K,2026-03-02,0001108524-26-000060
1,10-Q,2025-12-04,0001108524-25-000238
2,10-Q,2025-09-04,0001108524-25-000088
3,10-Q,2025-05-29,0001108524-25-000030
4,10-K,2025-03-05,0001108524-25-000006
5,10-Q,2024-12-04,0001108524-24-000034
6,10-Q,2024-08-29,0001108524-24-000022
7,10-Q,2024-05-30,0001108524-24-000009
8,10-K,2024-03-06,0001108524-24-000005
9,10-Q,2023-11-30,0001108524-23-000048


---
## 4. Gap Analysis — Dataset vs Agent Coverage

In [18]:
# Full coverage matrix: which datasets cover which Shield AI scenarios
gap_data = [
    ('NDA / Confidentiality',            '✅', '✅', '⚠️ ', '✅'),
    ('SaaS / Software License',          '✅', '✅', '✅', '✅'),
    ('Vendor / Supply Chain',            '✅', '✅', '✅', '✅'),
    ('Healthcare (HIPAA/BAA)',            '✅', '⚠️ ', '⚠️ ', '✅'),
    ('Finance (SOX/PCI-DSS)',            '❌', '⚠️ ', '✅', '✅'),
    ('Government (FAR/DFARS)',           '❌', '⚠️ ', '✅', '✅'),
    ('Employment / Non-compete',         '❌', '✅', '✅', '✅'),
    ('IP Assignment / License',          '❌', '✅', '✅', '✅'),
    ('Partnership / JV',                 '❌', '✅', '✅', '✅'),
    ('GDPR / CCPA International',        '✅', '⚠️ ', '✅', '✅'),
    ('Expired contract (date edge)',     '❌', '❌', '❌', '✅'),
    ('$0 / uncapped liability',          '❌', '✅', '❌', '✅'),
    ('Prompt injection (security)',      '✅', '❌', '❌', '✅'),
    ('Multi-party (3+ parties)',         '❌', '⚠️ ', '⚠️ ', '✅'),
    ('Amendment / addendum',            '❌', '❌', '⚠️ ', '✅'),
    ('Missing liability cap',            '❌', '✅', '❌', '✅'),
    ('Contradictory clauses',            '❌', '❌', '❌', '✅'),
]

df_gap = pd.DataFrame(gap_data, columns=[
    'Scenario', 'Demo (7)', 'CUAD (510)', 'EDGAR', 'Synthetic (to build)'
])
print('=== Coverage matrix: ✅ Full  ⚠️ Partial  ❌ Missing ===')
display(df_gap)

# Score coverage
score_map = {'✅': 2, '⚠️ ': 1, '❌': 0}
for col in ['Demo (7)', 'CUAD (510)', 'EDGAR', 'Synthetic (to build)']:
    score = df_gap[col].map(score_map).sum()
    max_score = len(df_gap) * 2
    print(f'  {col:<25} coverage score: {score}/{max_score}  ({score/max_score*100:.0f}%)')

=== Coverage matrix: ✅ Full  ⚠️ Partial  ❌ Missing ===


,Scenario,Demo (7),CUAD (510),EDGAR,Synthetic (to build)
0,NDA / Confidentiality,✅,✅,⚠️,✅
1,SaaS / Software License,✅,✅,✅,✅
2,Vendor / Supply Chain,✅,✅,✅,✅
3,Healthcare (HIPAA/BAA),✅,⚠️,⚠️,✅
4,Finance (SOX/PCI-DSS),❌,⚠️,✅,✅
5,Government (FAR/DFARS),❌,⚠️,✅,✅
6,Employment / Non-compete,❌,✅,✅,✅
7,IP Assignment / License,❌,✅,✅,✅
8,Partnership / JV,❌,✅,✅,✅
9,GDPR / CCPA International,✅,⚠️,✅,✅


  Demo (7)                  coverage score: 12/34  (35%)
  CUAD (510)                coverage score: 21/34  (62%)
  EDGAR                     coverage score: 20/34  (59%)
  Synthetic (to build)      coverage score: 34/34  (100%)


---
## 5. Synthetic Contract Plan — Fill the Remaining Gaps

In [19]:
synthetic_plan = [
    {
        'filename':        'Government_Procurement_FAR.pdf',
        'type':            'Government',
        'frameworks':      ['FAR', 'DFARS'],
        'expected_risk':   'Medium',
        'agent_tested':    'Agent 3',
        'scenario':        'FAR mandatory clauses present/missing → compliance fail',
        'has_injection':   False,
    },
    {
        'filename':        'Finance_PCI_DSS_Agreement.pdf',
        'type':            'Finance',
        'frameworks':      ['PCI-DSS', 'SOX'],
        'expected_risk':   'High',
        'agent_tested':    'Agent 3',
        'scenario':        'Payment processing — PCI-DSS scope 1 requirements must be flagged',
        'has_injection':   False,
    },
    {
        'filename':        'Zero_Liability_Cap.pdf',
        'type':            'Vendor',
        'frameworks':      [],
        'expected_risk':   'Critical',
        'agent_tested':    'Agent 2',
        'scenario':        'Vendor liability cap = $0 AND uncapped indemnification → score must be 90+',
        'has_injection':   False,
    },
    {
        'filename':        'Expired_Termination_Date.pdf',
        'type':            'Service',
        'frameworks':      [],
        'expected_risk':   'High',
        'agent_tested':    'Agent 1',
        'scenario':        'Contract expired 2 years ago — Agent 1 must extract and flag past termination date',
        'has_injection':   False,
    },
    {
        'filename':        'Multi_Party_4_Parties.pdf',
        'type':            'Partnership',
        'frameworks':      ['GDPR'],
        'expected_risk':   'Medium',
        'agent_tested':    'Agent 1',
        'scenario':        '4-party data sharing agreement — Agent 1 must extract all 4 parties correctly',
        'has_injection':   False,
    },
    {
        'filename':        'CSS_Hidden_Injection.pdf',
        'type':            'NDA',
        'frameworks':      [],
        'expected_risk':   'Critical',
        'agent_tested':    'Agent 4',
        'scenario':        'CSS white-on-white injection (different technique than Vendor_Agreement.pdf)',
        'has_injection':   True,
        'injection_type':  'css_hidden_text',
    },
    {
        'filename':        'Employment_Aggressive_NonCompete.pdf',
        'type':            'Employment',
        'frameworks':      [],
        'expected_risk':   'Medium',
        'agent_tested':    'Agent 7',
        'scenario':        'Aggressive 3-year non-compete + no-solicit — Agent 7 Q&A ground truth',
        'has_injection':   False,
    },
    {
        'filename':        'GDPR_CCPA_Dual_DPA.pdf',
        'type':            'Data Processing',
        'frameworks':      ['GDPR', 'CCPA'],
        'expected_risk':   'Medium',
        'agent_tested':    'Agent 3',
        'scenario':        'EU + California combined DPA — Agent 3 must check both frameworks simultaneously',
        'has_injection':   False,
    },
    {
        'filename':        'Contradictory_Clauses.pdf',
        'type':            'Vendor',
        'frameworks':      [],
        'expected_risk':   'High',
        'agent_tested':    'Agent 2',
        'scenario':        'Section 4 says "unlimited liability", Section 12 says "capped at $100" — risk agent edge case',
        'has_injection':   False,
    },
    {
        'filename':        'IP_Assignment_Heavy.pdf',
        'type':            'IP / Tech',
        'frameworks':      [],
        'expected_risk':   'High',
        'agent_tested':    'Agent 2 + 7',
        'scenario':        'All IP assigned to vendor, no license back — Agent 2 must flag, Agent 7 must answer clause questions',
        'has_injection':   False,
    },
]

df_syn = pd.DataFrame(synthetic_plan)
print(f'=== Synthetic contracts to generate: {len(df_syn)} ===')
display(df_syn[['filename', 'type', 'frameworks', 'expected_risk', 'agent_tested', 'scenario']])

out = DATA_DIR / 'synthetic_contract_plan.json'
out.write_text(json.dumps(synthetic_plan, indent=2))
print(f'\nPlan saved to {out}')

=== Synthetic contracts to generate: 10 ===


,filename,type,frameworks,expected_risk,agent_tested,scenario
0,Government_Procurement_FAR.pdf,Government,"[FAR, DFARS]",Medium,Agent 3,FAR mandatory clauses present/missing → compliance fail
1,Finance_PCI_DSS_Agreement.pdf,Finance,"[PCI-DSS, SOX]",High,Agent 3,Payment processing — PCI-DSS scope 1 requirements must be flagged
2,Zero_Liability_Cap.pdf,Vendor,[],Critical,Agent 2,Vendor liability cap = $0 AND uncapped indemnification → score must be 90+
3,Expired_Termination_Date.pdf,Service,[],High,Agent 1,Contract expired 2 years ago — Agent 1 must extract and flag past termination date
4,Multi_Party_4_Parties.pdf,Partnership,[GDPR],Medium,Agent 1,4-party data sharing agreement — Agent 1 must extract all 4 parties correctly
5,CSS_Hidden_Injection.pdf,NDA,[],Critical,Agent 4,CSS white-on-white injection (different technique than Vendor_Agreement.pdf)
6,Employment_Aggressive_NonCompete.pdf,Employment,[],Medium,Agent 7,Aggressive 3-year non-compete + no-solicit — Agent 7 Q&A ground truth
7,GDPR_CCPA_Dual_DPA.pdf,Data Processing,"[GDPR, CCPA]",Medium,Agent 3,EU + California combined DPA — Agent 3 must check both frameworks simultaneously
8,Contradictory_Clauses.pdf,Vendor,[],High,Agent 2,"Section 4 says ""unlimited liability"", Section 12 says ""capped at $100"" — risk agent edge case"
9,IP_Assignment_Heavy.pdf,IP / Tech,[],High,Agent 2 + 7,"All IP assigned to vendor, no license back — Agent 2 must flag, Agent 7 must answer clause questions"



Plan saved to /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data/synthetic_contract_plan.json


In [20]:
# Overall corpus plan visualisation
corpus_plan = pd.DataFrame([
    {'source': 'Existing demo contracts',    'count': len(demo_files),      'priority': 1},
    {'source': 'CUAD top 20 (real)',         'count': 20,                   'priority': 1},
    {'source': 'Synthetic (gap-filling)',    'count': len(synthetic_plan),  'priority': 2},
    {'source': 'SEC EDGAR (optional +10)',   'count': 10,                   'priority': 3},
])
corpus_plan['total_running'] = corpus_plan['count'].cumsum()
total = corpus_plan['count'].sum()

fig = px.bar(
    corpus_plan,
    x='source',
    y='count',
    color='source',
    text='count',
    title=f'Planned Shield AI corpus: {total} contracts total',
    template='plotly_dark',
    labels={'count': 'Contracts', 'source': 'Source'},
)
fig.update_traces(textposition='outside')
fig.show()

print(f'Total: {total} contracts')
print('Risk distribution target (after adding CUAD + synthetic):')
print('  Low risk (auto-approve) : ~10 contracts')
print('  Medium risk (review)    : ~15 contracts')
print('  High risk (legal)       : ~10 contracts')
print('  Critical / quarantined  : ~2 contracts')

Total: 47 contracts
Risk distribution target (after adding CUAD + synthetic):
  Low risk (auto-approve) : ~10 contracts
  Medium risk (review)    : ~15 contracts
  High risk (legal)       : ~10 contracts
  Critical / quarantined  : ~2 contracts


---
## 6. Summary & Recommended Next Steps

In [21]:
print('='*65)
print('SHIELD AI — DATASET EXPLORATION SUMMARY')
print('='*65)

cuad_top_types = top20['inferred_type'].value_counts().head(3)

print(f'''
CUAD master_clauses.csv:
  Total contracts : {len(df)}
  Clause types    : {len(clause_cols)} annotated categories
  Avg completeness: {df["completeness_pct"].mean():.0f}% of clause types present per contract
  Top types       : {dict(cuad_top_types)}
  Saved to        : {CUAD_CACHE}

Top 20 CUAD contracts selected for ingestion:
  → {DATA_DIR}/cuad_top20_for_ingestion.json
  → Selection criteria: most priority clauses (liability, audit rights,
    governing law, anti-assignment, non-compete, IP assignment)

Synthetic contracts plan (10 contracts):
  → {DATA_DIR}/synthetic_contract_plan.json
  → Covers: Government FAR, Finance PCI-DSS, $0 liability,
    expired dates, multi-party, CSS injection, employment,
    GDPR+CCPA, contradictory clauses, IP assignment

RECOMMENDED ACTIONS (in order):

  1. [Next notebook] 02_cuad_ingestion.ipynb
     Download top 20 CUAD contract TXT files and convert to PDF
     for upload to Shield AI backend

  2. [Next notebook] 03_synthetic_generator.ipynb  
     Use Gemini to generate the 10 synthetic contracts from the plan
     Each has known properties so we can measure agent accuracy

  3. [After ingestion] 04_agent_evaluation.ipynb
     Compare Agent 1/2/3 outputs against CUAD ground truth labels
     → e.g., did Agent 2 find the liability cap that CUAD annotated?
''')

print('All outputs saved to:', DATA_DIR)

SHIELD AI — DATASET EXPLORATION SUMMARY

CUAD master_clauses.csv:
  Total contracts : 510
  Clause types    : 42 annotated categories
  Avg completeness: 96% of clause types present per contract
  Top types       : {'Other / Mixed': np.int64(9), 'Distribution': np.int64(7), 'Service Agreement': np.int64(2)}
  Saved to        : /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data/master_clauses.csv

Top 20 CUAD contracts selected for ingestion:
  → /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data/cuad_top20_for_ingestion.json
  → Selection criteria: most priority clauses (liability, audit rights,
    governing law, anti-assignment, non-compete, IP assignment)

Synthetic contracts plan (10 contracts):
  → /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data/synthetic_contract_plan.json
  → Covers: Government FAR, Finance PCI-DSS, $0 liability,
    expired dates, multi-party, CSS injection, employment,
    GDPR+CC